In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
from io import StringIO
import pandas as pd
import numpy as np
import gcat as gc
import sciris as sc
import mnch
import gcat as gc
import matplotlib.pyplot as plt
import atomica as at
from collections import defaultdict 
import os

try:
    import dataframe_image as dfi
except:
    pass

import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 14,'figure.dpi':300})


idx = pd.IndexSlice # Helper for multi-index lookups

ModuleNotFoundError: No module named 'gcat'

In [ ]:
#Set region and scenario for analysis
Region = 'SA'
results_dir = mnch.root/'paper'/'results'
path = 'epi_df_'+ Region +'90.xlsx'
epi_df = pd.read_excel(results_dir / path)
epi_df = epi_df.set_index(['Scenario','Population Name','Year','Cause','Measure'])

In [3]:
#Check births output are consistent
num_births = epi_df['Value'].unstack()['births'].dropna().groupby(level='Scenario').sum()
num_births

NameError: name 'epi_df' is not defined

In [4]:
#Check incidences in each scenario
df = epi_df['Value'].unstack()['cases'].unstack().groupby(level='Scenario').sum()
df = df.drop('demographics',axis=1)

approx_incidence = df.divide(num_births, axis=0)
approx_incidence['hemorrhage']

Scenario
ai_ultrasound               0.092363
non_invasive_hb_test        0.092253
pe_screening                0.092363
soc                         0.092363
soc_diagnostics_scale_up    0.090530
soc_hb_test                 0.092346
soc_iron_tablets            0.091315
soc_pe_treatment            0.092363
soc_pe_urine_dipstick       0.092363
soc_rutf                    0.092363
soc_scale_up                0.091189
soc_ultrasound              0.092363
Name: hemorrhage, dtype: float64

In [5]:
#Check impact of interventions on incidence
df = approx_incidence/approx_incidence.loc['soc']
s = df.style.highlight_between(left=0.0, right=0.9999, inclusive='neither', color='#4287f5')  
s.highlight_between(left=1, right=np.inf, inclusive='neither', color='#ff0000')  

Cause,anemia,anemia_pp,hemorrhage,maternal sepsis,neonatal encephalopathy,neonatal sepsis,non-rds preterm,obstructed labour,other neonatal conditions,preeclampsia,preterm,preterm & sga,rds,sga,stillbirth,stunting,wasting,wasting 19 months to 59 months,wasting 6 months to 18 months,wasting 8d to 5 months
Scenario,,,,,,,,,,,,,,,,,,,,
ai_ultrasound,1.000000,1.000108,1.000000,1.000000,0.933352,1.000906,nan,1.000000,nan,1.000000,1.000076,1.000076,1.000906,1.000076,nan,1.000906,nan,nan,1.002072,1.002129
non_invasive_hb_test,0.995306,0.996600,0.998807,1.000000,1.000016,1.000004,nan,1.000000,nan,1.000000,1.000001,0.998197,1.000004,0.998197,nan,1.000004,nan,nan,1.000007,0.999332
pe_screening,1.000000,1.000000,1.000000,1.000000,1.000428,1.000416,nan,1.000000,nan,0.943471,0.987741,0.985700,0.988141,0.998920,nan,1.000416,nan,nan,1.000500,1.000112
soc,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,nan,1.000000,nan,1.000000,1.000000,1.000000,1.000000,1.000000,nan,1.000000,nan,nan,1.000000,1.000000
soc_diagnostics_scale_up,0.921597,0.943297,0.980156,1.000000,0.934077,1.001470,nan,1.000000,nan,0.929339,0.983786,0.951802,0.985136,0.968685,nan,1.001470,nan,nan,1.002800,0.991094
soc_hb_test,0.999273,0.999473,0.999815,1.000000,1.000003,1.000001,nan,1.000000,nan,1.000000,1.000000,0.999721,1.000001,0.999721,nan,1.000001,nan,nan,1.000001,0.999896
soc_iron_tablets,0.955219,0.967606,0.988655,1.000000,1.000154,1.000037,nan,1.000000,nan,1.000000,1.000006,0.982820,1.000037,0.982820,nan,1.000037,nan,nan,1.000068,0.993646
soc_pe_treatment,1.000000,1.000000,1.000000,1.000000,1.000002,1.000002,nan,1.000000,nan,1.000000,0.999794,0.999778,0.999796,1.000001,nan,1.000002,nan,nan,1.000004,1.000004
soc_pe_urine_dipstick,1.000000,1.000000,1.000000,1.000000,1.000003,1.000002,nan,1.000000,nan,1.000000,0.999774,0.999757,0.999777,1.000001,nan,1.000002,nan,nan,1.000004,1.000005


In [6]:
#Check intervention impact on cases
epi_df_2040 = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()
df = epi_df_2040['cases'].unstack().groupby(level='Scenario').sum()
df = df.drop('demographics',axis=1)

df

df = df/df.loc['soc']
df = df.dropna(how='all', axis=1)
df = df.sort_index(axis=1)
df=df.T

s = df.style
s = s.highlight_between(left=0.0, right=0.9999, inclusive='neither', color='#ccdcfc') 
s = s.highlight_between(left=0.0, right=0.99, inclusive='neither', color='#4287f5') 
s = s.highlight_between(left=1, right=np.inf, inclusive='neither', color='#ffdddd')  
s = s.highlight_between(left=1.01, right=np.inf, inclusive='neither', color='#ff0000')  
s = s.format(lambda x: f'{(x-1)*100:+.2g}%' if not np.isnan(x) else '-')
s.set_table_styles([
    {"selector": "td,th", "props": "height: 3em;"},
    {"selector": "th", "props": [("border", "1px solid")]}   
])
s = s.set_properties(**{'text-align': 'center','border':'1px solid black'})

try:
    dfi.export(s,str(fig_dir/'daly_impact.png'))
except:
    pass

s

Scenario,ai_ultrasound,non_invasive_hb_test,pe_screening,soc,soc_diagnostics_scale_up,soc_hb_test,soc_iron_tablets,soc_pe_treatment,soc_pe_urine_dipstick,soc_rutf,soc_scale_up,soc_ultrasound
Cause,,,,,,,,,,,,
anemia,+0%,-0.54%,+0%,+0%,-8.6%,-0.087%,-4.9%,+0%,+0%,+0%,-5.5%,+0%
anemia_pp,+0.013%,-0.39%,-5.9e-06%,+0%,-6.2%,-0.062%,-3.5%,-8.4e-08%,-9.4e-08%,+0%,-3.9%,+0.0053%
hemorrhage,+0%,-0.14%,+0%,+0%,-2.2%,-0.022%,-1.2%,+0%,+0%,+0%,-1.4%,+0%
maternal sepsis,+0%,+0%,+0%,+0%,+0%,+0%,+0%,+0%,+0%,+0%,+0%,+0%
neonatal encephalopathy,-7.8%,+0.002%,+0.04%,+0%,-7.7%,+0.00031%,+0.017%,+0.00028%,+0.00031%,+0%,-3.3%,-3.4%
neonatal sepsis,+0.1%,+0.00045%,+0.038%,+0%,+0.16%,+7.1e-05%,+0.004%,+0.00025%,+0.00028%,+0%,+0.048%,+0.043%
obstructed labour,+0%,+0%,+0%,+0%,+0%,+0%,+0%,+0%,+0%,+0%,+0%,+0%
preeclampsia,+0%,+0%,-5.2%,+0%,-7.9%,+0%,+0%,+0%,+0%,+0%,+0%,+0%
preterm,+0.0089%,+7.6e-05%,-1.1%,+0%,-1.8%,+1.2e-05%,+0.00067%,-0.023%,-0.025%,+0%,-0.058%,+0.0037%


In [7]:
epi_df_2040 = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()
df = epi_df_2040['cases'].unstack().groupby(level='Scenario').sum()

In [4]:
#Check intervention impact on DALYs
df = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum()
df = df.drop('demographics',axis=1)

df

df = df/df.loc['soc']
df = df.dropna(how='all', axis=1)
df = df.sort_index(axis=1)
df = df.T

s = df.style
s = s.highlight_between(left=0.0, right=0.9999, inclusive='neither', color='#ccdcfc') 
s = s.highlight_between(left=0.0, right=0.99, inclusive='neither', color='#4287f5') 
s = s.highlight_between(left=1, right=np.inf, inclusive='neither', color='#ffdddd')  
s = s.highlight_between(left=1.01, right=np.inf, inclusive='neither', color='#ff0000')  
s = s.format(lambda x: f'{(x-1)*100:+.2g}%' if not np.isnan(x) else '-')
s.set_table_styles([
    {"selector": "td,th", "props": "height: 3em;"},
    {"selector": "th", "props": [("border", "1px solid")]}   
])
s = s.set_properties(**{'text-align': 'center','border':'1px solid black'})

try:
    dfi.export(s,str(fig_dir/'daly_impact.png'))
except:
    pass

s

NameError: name 'epi_df_2040' is not defined

In [9]:
#Check DALYs attributable to conditions
df = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum()
df = df.drop('demographics',axis=1)

df

df = df.loc['soc']
df

Cause
anemia                            1.304420e+07
anemia_pp                         4.540657e+07
hemorrhage                        2.179548e+07
maternal sepsis                   3.262837e+06
neonatal encephalopathy           2.790698e+08
neonatal sepsis                   8.027514e+07
non-rds preterm                   1.336991e+08
obstructed labour                 5.072621e+06
other neonatal conditions         9.072596e+07
preeclampsia                      1.107742e+07
preterm                           0.000000e+00
preterm & sga                     0.000000e+00
rds                               1.975593e+08
sga                               1.745649e+08
stillbirth                        6.495644e+08
stunting                          2.519587e+07
wasting                           2.407255e+08
wasting 19 months to 59 months    0.000000e+00
wasting 6 months to 18 months     0.000000e+00
wasting 8d to 5 months            0.000000e+00
Name: soc, dtype: float64

In [5]:
#DALYs averted check
epi_df_2040 = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()
dalys = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum()
dalys_averted = dalys.loc['soc'] - dalys
dalys_averted

NameError: name 'epi_df' is not defined

In [16]:
##OUTPUTS SECTION##
#This section takes the epi outcomes and formats them for the tables in the paper
regions = ['Other', 'SA','SSA']
scenarios = ['90']
shape = np.shape(epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm']))

    #work out how to separate indirect and direct
for scenario in scenarios:
    soc_indirect = []
    dalys_averted_sum = []
    outputs_1 = pd.DataFrame()
    outputs_2 = pd.DataFrame()
    outputs_3 = pd.DataFrame()
    outputs_4 = pd.DataFrame()
    dalys_averted_out = pd.DataFrame()
    soc_out = pd.DataFrame()
    soc_scale_up_out = pd.DataFrame()
    soc_diagnostics_scale_up_out = pd.DataFrame()
    soc_dalys = np.zeros(shape)
    soc_cases = np.zeros(shape)
    soc_deaths = np.zeros(shape)
    soc_scale_up_dalys = np.zeros(shape)
    soc_scale_up_cases = np.zeros(shape)
    soc_scale_up_deaths = np.zeros(shape)
    soc_diagnostics_scale_up_dalys = np.zeros(shape)
    soc_diagnostics_scale_up_cases = np.zeros(shape)
    soc_diagnostics_scale_up_deaths = np.zeros(shape)
    for region in regions:
        path = f'epi_df_{region}_{scenario}.xlsx'
        epi_df = pd.read_excel(results_dir / path)
        epi_df = epi_df.set_index(['Scenario', 'Population Name', 'Year', 'Cause', 'Measure'])
        epi_df_2040 = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()
        dalys = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum()
        dalys_averted = dalys.loc['soc'] - dalys

        # Calculate total DALYs


        # Calculate direct impacts
        direct_impacts_data = {
            region: [
                dalys_averted.loc['soc_rutf', ['wasting', 'wasting 19 months to 59 months', 'wasting 6 months to 18 months', 'wasting 8d to 5 months']].sum(axis=0),
                dalys_averted.loc['soc_iron_tablets', ['anemia']].sum(axis=0),
                dalys_averted.loc['soc_hb_test', ['anemia']].sum(axis=0),
                dalys_averted.loc['soc_pe_urine_dipstick', ['preeclampsia']].sum(axis=0),
                dalys_averted.loc['soc_ultrasound', ['obstructed labour', 'hemorrhage', 'preterm', 'non-rds preterm']].sum(axis=0),
                dalys_averted.loc['soc_pe_treatment', []].sum(axis=0),
                dalys_averted.loc['non_invasive_hb_test', ['anemia']].sum(axis=0),
                dalys_averted.loc['pe_screening', ['preeclampsia']].sum(axis=0),
                dalys_averted.loc['ai_ultrasound', ['obstructed labour', 'hemorrhage', 'preterm', 'non-rds preterm']].sum(axis=0)
            ]
        }
        direct_impacts = pd.DataFrame(direct_impacts_data)

        # Calculate DALYs averted sum
        dalys_averted_sum_data = {
            region: [
                dalys_averted.loc['soc_rutf'].sum(axis=0),
                dalys_averted.loc['soc_iron_tablets'].sum(axis=0),
                dalys_averted.loc['soc_hb_test'].sum(axis=0),
                dalys_averted.loc['soc_pe_urine_dipstick'].sum(axis=0),
                dalys_averted.loc['soc_ultrasound'].sum(axis=0),
                dalys_averted.loc['soc_pe_treatment'].sum(axis=0),
                dalys_averted.loc['non_invasive_hb_test'].sum(axis=0),
                dalys_averted.loc['pe_screening'].sum(axis=0),
                dalys_averted.loc['ai_ultrasound'].sum(axis=0)
            ]
        }
        dalys_averted_sum = pd.DataFrame(dalys_averted_sum_data)

        # Calculate indirect impacts
        indirect_impacts = dalys_averted_sum - direct_impacts

        # Adjust index
        direct_impacts.index = direct_impacts.index * 2
        indirect_impacts.index = indirect_impacts.index * 2 + 1

        # Concatenate direct and indirect impacts
        dalys_averted_out_region = pd.concat([direct_impacts, indirect_impacts]).sort_index()

        # Append total DALYs
        soc_all = pd.DataFrame({region: [dalys.loc['soc'].sum()]})
        soc_scale_up_total_averted = pd.DataFrame({region: [dalys_averted.loc['soc_scale_up'].sum()]})
        soc_diagnostics_scale_up_total_averted = pd.DataFrame({region: [dalys_averted.loc['soc_diagnostics_scale_up'].sum()]})
        dalys_averted_out_region = soc_all.append([dalys_averted_out_region, soc_scale_up_total_averted, soc_diagnostics_scale_up_total_averted])

        # Concatenate with previous results
        dalys_averted_out = pd.concat([dalys_averted_out, dalys_averted_out_region], axis=1)


        ## Outputs for key scenarios##
        soc_dalys_region = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm'])
        soc_cases_region = epi_df_2040['cases'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm'])
        soc_deaths_region = epi_df_2040['maternal deaths'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm']) + epi_df_2040['neonatal deaths'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm'])+epi_df_2040['child deaths'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm'])+epi_df_2040['neonatal  deaths'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm'])+epi_df_2040['stillbirth deaths'].unstack().groupby(level='Scenario').sum().loc['soc'].drop(['demographics', 'preterm & sga', 'preterm'])


        soc_scale_up_dalys_region = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum().loc['soc_scale_up'].drop(['demographics', 'preterm & sga', 'preterm'])
        soc_scale_up_cases_region = epi_df_2040['cases'].unstack().groupby(level='Scenario').sum().loc['soc_scale_up'].drop(['demographics', 'preterm & sga', 'preterm'])
        soc_scale_up_deaths_region = (epi_df_2040['maternal deaths'].unstack().groupby(level='Scenario').sum().loc['soc_scale_up']+ epi_df_2040['neonatal deaths'].unstack().groupby(level='Scenario').sum().loc['soc_scale_up']+epi_df_2040['child deaths'].unstack().groupby(level='Scenario').sum().loc['soc_scale_up']+epi_df_2040['neonatal  deaths'].unstack().groupby(level='Scenario').sum().loc['soc_scale_up']+epi_df_2040['stillbirth deaths'].unstack().groupby(level='Scenario').sum().loc['soc']).drop(['demographics', 'preterm & sga', 'preterm'])
        #soc_scale_up_region = {'Dalys'+region:soc_scale_up_dalys,
        #            'Cases'+region:soc_scale_up_cases,
        #           'Deaths'+region:soc_scale_up_deaths}
        
        soc_diagnostics_scale_up_dalys_region = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum().loc['soc_diagnostics_scale_up'].drop(['demographics', 'preterm & sga', 'preterm'])
        soc_diagnostics_scale_up_cases_region = epi_df_2040['cases'].unstack().groupby(level='Scenario').sum().loc['soc_diagnostics_scale_up'].drop(['demographics', 'preterm & sga', 'preterm'])
        soc_diagnostics_scale_up_deaths_region = (epi_df_2040['maternal deaths'].unstack().groupby(level='Scenario').sum().loc['soc_diagnostics_scale_up']+ epi_df_2040['neonatal deaths'].unstack().groupby(level='Scenario').sum().loc['soc_diagnostics_scale_up']+epi_df_2040['child deaths'].unstack().groupby(level='Scenario').sum().loc['soc_diagnostics_scale_up']+epi_df_2040['neonatal  deaths'].unstack().groupby(level='Scenario').sum().loc['soc_diagnostics_scale_up']+epi_df_2040['stillbirth deaths'].unstack().groupby(level='Scenario').sum().loc['soc']).drop(['demographics', 'preterm & sga', 'preterm'])
        # soc_diagnostics_scale_up_region = {'Dalys'+region:soc_scale_up_dalys,
        #            'Cases'+region:soc_diagnostics_scale_up_cases,
        #           'Deaths'+region:soc_diagnostics_scale_up_deaths}
        
        # #Sum regions together
        soc_dalys = soc_dalys + soc_dalys_region
        soc_cases = soc_cases +  soc_cases_region
        soc_deaths = soc_deaths + soc_deaths_region

        soc_scale_up_dalys = soc_scale_up_dalys + soc_scale_up_dalys_region
        soc_scale_up_cases = soc_scale_up_cases + soc_scale_up_cases_region
        soc_scale_up_deaths = soc_scale_up_deaths +soc_scale_up_deaths_region

        soc_diagnostics_scale_up_dalys = soc_diagnostics_scale_up_dalys + soc_diagnostics_scale_up_dalys_region
        soc_diagnostics_scale_up_cases = soc_diagnostics_scale_up_cases + soc_diagnostics_scale_up_cases_region
        soc_diagnostics_scale_up_deaths = soc_diagnostics_scale_up_deaths + soc_diagnostics_scale_up_deaths_region

        ##Outputs for key outputs table

        maternal_mortalities = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()['maternal deaths'].dropna().groupby(level='Scenario').sum()
        maternal_mortalities_averted_1 = maternal_mortalities["soc"]-maternal_mortalities["soc_scale_up"]
        maternal_mortalities_averted_2 = maternal_mortalities["soc"]-maternal_mortalities["soc_diagnostics_scale_up"]
        child_mortalities = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()['child deaths'].dropna().groupby(level='Scenario').sum()
        child_mortalities_averted_1 = child_mortalities["soc"]-child_mortalities["soc_scale_up"]
        child_mortalities_averted_2 = child_mortalities["soc"]-child_mortalities["soc_diagnostics_scale_up"]
        neonatal_mortalities = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()['neonatal deaths'].dropna().groupby(level='Scenario').sum()
        neonatal_mortalities_averted_1 = neonatal_mortalities["soc"]-neonatal_mortalities["soc_scale_up"]
        neonatal_mortalities_averted_2 = neonatal_mortalities["soc"]-neonatal_mortalities["soc_diagnostics_scale_up"]
        stillbirth_mortalities = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()['stillbirth deaths'].dropna().groupby(level='Scenario').sum()
        stillbirth_mortalities_averted_1 = stillbirth_mortalities["soc"]-stillbirth_mortalities["soc_scale_up"]
        stillbirth_mortalities_averted_2 = stillbirth_mortalities["soc"]-stillbirth_mortalities["soc_diagnostics_scale_up"]
        dalys = epi_df['Value'].loc[(epi_df.index.get_level_values('Year')>=2025)&(epi_df.index.get_level_values('Year')<=2040)].unstack()['dalys'].dropna().groupby(level='Scenario').sum()
        dalys_averted_1 = dalys["soc"]-dalys["soc_scale_up"]
        dalys_averted_2 = dalys["soc"]-dalys["soc_diagnostics_scale_up"]

        df_region = epi_df_2040['dalys'].unstack().groupby(level='Scenario').sum()
        df_region = df_region.drop('demographics',axis=1)
        df_region = df_region.loc['soc'] - df_region
        df_region = df_region.dropna(how='all', axis=1).sort_index(axis=1).T

        outputs_1_region = pd.DataFrame({
            'Maternal mortalities averted': maternal_mortalities_averted_1,
            'Child mortalities averted': child_mortalities_averted_1,
            'Neonatal mortalities averted': neonatal_mortalities_averted_1,
            'Stillbirth mortalities averted': stillbirth_mortalities_averted_1,
            'Dalys averted': dalys_averted_1
        }, index = [region])

        outputs_2_region = pd.DataFrame({
            'Maternal mortalities averted': maternal_mortalities_averted_2,
            'Child mortalities averted': child_mortalities_averted_2,
            'Neonatal mortalities averted': neonatal_mortalities_averted_2,
            'Stillbirth mortalities averted': stillbirth_mortalities_averted_2,
            'Dalys averted': dalys_averted_2
        }, index = [region])

        outputs_3_region = pd.DataFrame({
            'Maternal mortalities averted': maternal_mortalities["soc"],
            'Child mortalities averted': child_mortalities["soc"],
            'Neonatal mortalities averted': neonatal_mortalities["soc"],
            'Stillbirth mortalities averted': stillbirth_mortalities["soc"],
            'Dalys averted': dalys["soc"]
        }, index = [region])
        
        outputs_1 = pd.concat([outputs_1, outputs_1_region])
        outputs_2 = pd.concat([outputs_2, outputs_2_region])
        outputs_3 = pd.concat([outputs_3, outputs_3_region])
        outputs_4 = outputs_4.add(df_region, fill_value=0)

    index = ['soc_total', 'soc_rutf_direct', 'soc_rutf_indirect', 'soc iron tablets direct', 'soc iron tablets indirect', 'SOC Hb test direct', 'SOC Hb test indirect', 'SOC pre-eclampsia UDS direct', 'SOC pre-eclampsia UDS indirect', 
             'SOC ultrasound direct','SOC ultrasound indirect', 'soc preeclampsia treatment direct', 'soc preeclampsia treatment indirect',
            'non invasive hb test direct', 'non invasive hb test indirect', 'preeclampsia screening and treatment direct', 'preeclampsia screening and treatment indirect', 'AI ultrasound direct', 'AI ultrasound indirect', 'SOC scale up', 'SOC + diagnostics scale up']
    dalys_averted_out = dalys_averted_out.set_index([pd.Index(index)])
    soc_out = pd.concat([soc_dalys, soc_cases, soc_deaths], axis = 1)
    soc_scale_up_out = pd.concat([soc_scale_up_dalys, soc_scale_up_cases, soc_scale_up_deaths], axis = 1)
    soc_diagnostics_scale_up_out = pd.concat([soc_diagnostics_scale_up_dalys, soc_diagnostics_scale_up_cases, soc_diagnostics_scale_up_deaths], axis = 1)
    soc_out.columns = ["soc_dalys", "soc_cases", "soc_deaths"]
    soc_scale_up_out.columns = ["dalys", "cases", "deaths"]
    soc_diagnostics_scale_up_out.columns = ["dalys", "cases", "deaths"]
    #Create totals row
    dalys_averted_out["Total"] = dalys_averted_out.sum(axis=1)
    outputs_1.loc["Total"] = outputs_1.sum()
    outputs_2.loc["Total"] = outputs_2.sum()
    outputs_3.loc["Total"] = outputs_3.sum()


C:\Users\Tom.walsh\AppData\Local\Temp\ipykernel_4836\4123935895.py:83: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dalys_averted_out_region = soc_all.append([dalys_averted_out_region, soc_scale_up_total_averted, soc_diagnostics_scale_up_total_averted])
C:\Users\Tom.walsh\AppData\Local\Temp\ipykernel_4836\4123935895.py:83: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dalys_averted_out_region = soc_all.append([dalys_averted_out_region, soc_scale_up_total_averted, soc_diagnostics_scale_up_total_averted])
C:\Users\Tom.walsh\AppData\Local\Temp\ipykernel_4836\4123935895.py:83: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dalys_averted_out_region = soc_all.append([dalys_averted_out_region, soc_scale_up_total_averted, soc_diagn

In [17]:
dalys_averted_out_formatted = dalys_averted_out.applymap("{:,.0f}".format)
soc_out_formatted = (soc_out/1000000).applymap("{:,.1f}".format)
soc_scale_up_out_formatted = (soc_scale_up_out/1000000).applymap("{:,.1f}".format)
soc_diagnostics_scale_up_out_formatted = (soc_diagnostics_scale_up_out/1000000).applymap("{:,.1f}".format)
outputs_1_formatted = outputs_1.applymap("{:,.0f}".format)
outputs_2_formatted = outputs_2.applymap("{:,.0f}".format)
outputs_3_formatted = outputs_3.applymap("{:,.0f}".format)
outputs_4_formatted = outputs_4.applymap("{:,.0f}".format)
name = "analysis_90_20250529.csv"
with pd.ExcelWriter(results_dir / name) as writer:
    dalys_averted_out_formatted.to_excel(writer, sheet_name='dalys averted by scenario')
    soc_out_formatted.to_excel(writer, sheet_name = 'SOC by cause')
    soc_scale_up_out_formatted.to_excel(writer, sheet_name = 'SOC SU by cause')
    soc_diagnostics_scale_up_out_formatted.to_excel(writer, sheet_name = 'SOC and diagnostics SU by cause')
    outputs_1_formatted.to_excel(writer, sheet_name='key outputs averted SOC')
    outputs_2_formatted.to_excel(writer, sheet_name='key outputs averted SOC+ Diag')
    outputs_3_formatted.to_excel(writer, sheet_name='baseline')
    outputs_4_formatted.to_excel(writer, sheet_name='daly reductions')